In [4]:
import gymnasium as gym

env = gym.make("Taxi-v4")

print("Environment:", env.spec.id)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

Environment: Taxi-v4
Observation space: Discrete(500)
Action space: Discrete(6)


In [5]:
observation, info = env.reset(seed=42)

print("Initial observation:", observation)
print("Action mask:", info["action_mask"])
print("Number of actions:", len(info["action_mask"]))

Initial observation: 386
Action mask: [1 1 0 1 0 0]
Number of actions: 6


In [6]:
env = gym.make("Taxi-v4", render_mode="ansi")

observation, info = env.reset(seed=42)

print(env.render())

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




In [7]:
print("Actions:")
for action, name in enumerate(
    ["South", "North", "East", "West", "Pickup", "Dropoff"]
):
    print(f"{action}: {name}")

observation, info = env.reset(seed=42)

action = 0
observation, reward, terminated, truncated, info = env.step(action)

print("\nAction taken:", action)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print("New observation:", observation)
print("New action mask:", info["action_mask"])

Actions:
0: South
1: North
2: East
3: West
4: Pickup
5: Dropoff

Action taken: 0
Reward: -1
Terminated: False
Truncated: False
New observation: 486
New action mask: [0 1 0 1 0 0]


## 1. Environment and Problem Setup

### 1.1 Selected Environment

For this project, we propose using **Taxi-v4** from the Gymnasium library to evaluate Maskable Proximal Policy Optimization (Maskable PPO).

Taxi-v4 is a discrete reinforcement learning environment in which an agent controls a taxi on a 5 × 5 grid. The agent must navigate to a passenger's location, pick up the passenger, and transport them to a specified destination.

The environment is suitable for studying action masking because not all actions are valid in every state. Action masks allow the agent to avoid selecting invalid actions during decision-making.

### 1.2 Observation Space

The observation space is `Discrete(500)`, representing 500 possible encoded states.

Each state contains information about:
- Taxi position (row and column)
- Passenger location
- Passenger destination

The environment returns the current state as an integer. In our initial experiment, the observation was `386`.

### 1.3 Action Space

The action space is `Discrete(6)` and consists of six actions:

| Action | Description |
|---|---|
| 0 | Move south |
| 1 | Move north |
| 2 | Move east |
| 3 | Move west |
| 4 | Pick up passenger |
| 5 | Drop off passenger |

### 1.4 Reward Function

The environment uses the following reward structure:

- **-1** for each ordinary step
- **+20** for a successful passenger drop-off
- **-10** for an illegal pickup or drop-off

This reward structure encourages the agent to complete the task efficiently while avoiding unnecessary or invalid actions.

### 1.5 Episode Termination

An episode terminates when the passenger is successfully delivered to the destination.

An episode may also be truncated when the environment's maximum step limit is reached.

### 1.6 Action Masking

Taxi-v4 provides an action mask indicating which actions are currently valid.

During initialization with `seed=42`, the environment returned:

`[1, 1, 0, 1, 0, 0]`

This means that actions South, North, and West were available, while East, Pickup, and Dropoff were masked.

After executing action South, the mask changed to:

`[0, 1, 0, 1, 0, 0]`

This demonstrates that valid actions depend on the current state.

### 1.7 Suitability for Maskable PPO

Taxi-v4 is an appropriate environment for evaluating Maskable PPO because:

1. It has a discrete action space compatible with Maskable PPO.
2. It provides state-dependent action masks.
3. Its relatively small state space makes experiments computationally manageable.
4. It allows comparisons between masked and unmasked agents.
5. Its rewards and episode outcomes are straightforward to interpret.

The environment therefore provides a practical setting for investigating how invalid action masking affects reinforcement learning performance.

### 1.8 Comparison of Candidate Environments

Three environments were considered for the project:

**Taxi-v4 (Gymnasium)**

A grid-based navigation task with 500 discrete states and
6 possible actions. The environment provides state-dependent
action masks and is easy to visualize and evaluate.

**InvalidActionEnvDiscrete (sb3-contrib)**

A simple environment designed to demonstrate invalid action
masking. It is useful for testing Maskable PPO implementations,
but its task is less realistic and less interpretable.

**Connect Four**

A turn-based board game in which placing a piece in a full
column is an invalid action. It provides an interesting
application of action masking but requires a more complex
environment setup and training procedure.

**Proposed choice: Taxi-v4**

Taxi-v4 offers the best balance between simplicity,
interpretability, computational efficiency, and support
for action masking.

The final environment selection will be confirmed
with the project team.

In [9]:
import numpy as np
import gymnasium as gym

from sb3_contrib import MaskablePPO
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker

def mask_fn(env):
    return env.unwrapped.action_mask(env.unwrapped.s)

masked_env = ActionMasker(
    gym.make("Taxi-v4"),
    mask_fn
)

observation, info = masked_env.reset(seed=42)

mask = get_action_masks(masked_env)

print("Observation space:", masked_env.observation_space)
print("Action space:", masked_env.action_space)
print("Action mask:", mask)
print("Mask shape:", mask.shape)
print("Maskable PPO imported:", MaskablePPO.__name__)

assert mask.shape == (masked_env.action_space.n,)
assert np.any(mask)

print("Compatibility check passed!")

Observation space: Discrete(500)
Action space: Discrete(6)
Action mask: [1 1 0 1 0 0]
Mask shape: (6,)
Maskable PPO imported: MaskablePPO
Compatibility check passed!


### 1.9 Environment Compatibility Verification

The selected Taxi-v4 environment was successfully initialized
using Gymnasium and tested for compatibility with the
action-masking interface provided by sb3-contrib.

The following properties were verified:

- Observation space: Discrete(500)
- Action space: Discrete(6)
- Initial observation (seed=42): 386
- Initial action mask: [1, 1, 0, 1, 0, 0]
- Action mask shape: (6,)
- At least one valid action is available.

The environment was wrapped using the ActionMasker wrapper,
and the action mask was successfully retrieved using
get_action_masks() from sb3-contrib.

The compatibility check passed without errors.

This confirms that Taxi-v4 can provide valid action masks
through the interface expected by Maskable PPO.

Model training and performance evaluation will be performed
in subsequent project stages.